# Prepare toots for OpenAI Moderator API

The OpenAI moderator API can processs requests in batches of up to 50,000 items. We pack of toots into batches, so that toots per instance stay together. However, as the distribution of toots per instance is heavy-tailed, we randomly sample down toots from large instances.

OpenAI Moderator requires a few things for batchwise processing:
1. per Batch, every request needs a unique identifier
2. we need to be able to map back the moderator response to the original toot and instance, so instance name needs to be part of the unique identifier
3. Empty toots need to be removed

In [1]:
import pandas as pd
from os.path import join
from glob import glob
from config import DATA_DIR
from helpers import clean_toots, clean_url
from hs_helpers import prepare_batch_input

In [2]:
# all available toots per instance
instances = glob(join(DATA_DIR, "toots_per_instance", "*.parquet")) 

In [3]:
df_list = []
for instance in instances:
    df = pd.read_parquet(instance)
    df_list.append(df)
    if len(df_list)%1000 == 0:
        print(f"Loaded {len(df_list)} instances...")

Loaded 1000 instances...
Loaded 2000 instances...
Loaded 3000 instances...
Loaded 4000 instances...
Loaded 5000 instances...
Loaded 6000 instances...
Loaded 7000 instances...
Loaded 8000 instances...
Loaded 9000 instances...
Loaded 10000 instances...


In [4]:
df = pd.concat(df_list, ignore_index=True)

In [5]:
# total number of toots
len(df)

14020291

In [7]:
del df_list

In [6]:
df = df[["id", "instance", "cleaned_content"]]

In [7]:
df.groupby("instance").size().sort_values(ascending=False).head()

instance
social.trom.tf          10039
veenus.art              10039
pl.slash.cl             10039
friendica.eskimo.com    10039
squeet.me               10038
dtype: int64

In [8]:
df.groupby("instance").size().describe()

count    10155.000000
mean      1380.629345
std       2890.304300
min          1.000000
25%         16.000000
50%        104.000000
75%        774.500000
max      10039.000000
dtype: float64

In [9]:
df.instance.nunique()

10155

In [10]:
# Create a unique identifier for each toot which contains the instance name, so that after OpenAI moderator responses are created, we can match them back to the instances 
df["instance_id"] = df.apply(lambda row: f"{row['instance']}_{row['id']}", axis=1)

In [11]:
len(df)

14020291

In [12]:
df.dropna(subset=["cleaned_content"], inplace=True)
df = df[df["cleaned_content"].astype(str).str.strip() != ""]

In [13]:
len(df)

13704942

Removing empty toots has a minor effect on the total number of toots, but it still removes around 300,000 toots.

In [14]:
df.drop_duplicates(subset=["instance_id"], inplace=True)

In [15]:
df.groupby("instance").size().describe()

count    10137.000000
mean      1344.885272
std       2825.147613
min          1.000000
25%         15.000000
50%        101.000000
75%        752.000000
max      10039.000000
dtype: float64

In [16]:
df.shape

(13633102, 4)

In [17]:
df["instance"].nunique()

10137

## Sampling
As the distribution of toots per instance is heavy-tailed, we sample 752 toots (75-percentile) from each instance which has more than that.

In [18]:
def sample_instance(group, max_size=752):
    if len(group) > max_size:
        return group.sample(max_size, random_state=42)
    else:
        return group

In [19]:
df_sampled = df.groupby("instance").apply(sample_instance).reset_index(drop=True)

/var/folders/5r/147qn_r949n270dn6qvhgnrw0000gn/T/ipykernel_25734/4112340785.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sampled = df.groupby("instance").apply(sample_instance).reset_index(drop=True)


In [20]:
len(df_sampled)

2814985

In [21]:
df_sampled = df_sampled[["instance", "instance_id", "cleaned_content"]]

In [22]:
df_sampled.instance_id.nunique()

2814985

In [23]:
def create_instance_based_batches(data, max_batch_size=47000):
    "Split dataframe into smaller ones, while keeping toots from the same instance together."
    instance_to_instance_size = dict(df_sampled.groupby("instance").size().astype(int))
    instance_to_instance_size_sorted = dict(sorted(instance_to_instance_size.items(), key=lambda item: item[1], reverse=True))
    instance_batches = []
    current_batch = []
    current_batch_size = 0
    # max_batch_size = 47000 # Leave some buffer below 50,000
    for instance, size in instance_to_instance_size_sorted.items():
        if current_batch_size + size > max_batch_size:
            instance_batches.append(current_batch)
            current_batch = [instance]
            current_batch_size = size
        else:
            current_batch.append(instance)
            current_batch_size += size
    if current_batch:
        instance_batches.append(current_batch)
    return instance_batches

In [24]:
instance_batches = create_instance_based_batches(df_sampled, max_batch_size=47000)

In [25]:
len(instance_batches)

61

In [26]:
len(instance_batches[0])

62

## Serialize into JSON batches

In [71]:
for i, batch in enumerate(instance_batches):
    df_batch = df_sampled[df_sampled["instance"].isin(batch)]
    if df_batch.instance_id.nunique() != len(df_batch):
        raise ValueError("Duplicate instance_id in batch!")
    prepare_batch_input(df_batch, batch_no=i, text_col="cleaned_content", id_col="instance_id")
    print(f"Prepared batch {i} with {len(df_batch)} toots.")

Prepared batch 0 with 46624 toots.
Prepared batch 1 with 46624 toots.
Prepared batch 2 with 46624 toots.
Prepared batch 3 with 46624 toots.
Prepared batch 4 with 46624 toots.
Prepared batch 5 with 46624 toots.
Prepared batch 6 with 46624 toots.
Prepared batch 7 with 46624 toots.
Prepared batch 8 with 46624 toots.
Prepared batch 9 with 46624 toots.
Prepared batch 10 with 46624 toots.
Prepared batch 11 with 46624 toots.
Prepared batch 12 with 46624 toots.
Prepared batch 13 with 46624 toots.
Prepared batch 14 with 46624 toots.
Prepared batch 15 with 46624 toots.
Prepared batch 16 with 46624 toots.
Prepared batch 17 with 46624 toots.
Prepared batch 18 with 46624 toots.
Prepared batch 19 with 46624 toots.
Prepared batch 20 with 46624 toots.
Prepared batch 21 with 46624 toots.
Prepared batch 22 with 46624 toots.
Prepared batch 23 with 46624 toots.
Prepared batch 24 with 46624 toots.
Prepared batch 25 with 46624 toots.
Prepared batch 26 with 46624 toots.
Prepared batch 27 with 46624 toots.
Pr

In [73]:
df_sampled.to_csv(join(DATA_DIR, "toots_per_instance_sampled_75_percentile.csv"), index=False)

## Serialize samples for manual evaluation

### Random sample 

In [62]:
random_sample_batch = df_sampled.sample(49000, random_state=42)

In [72]:
prepare_batch_input(random_sample_batch, batch_no="random_sample", text_col="cleaned_content", id_col="instance_id")

### Sample from moderated servers 

In [6]:
# Create a random sample
moderated_toots = pd.read_csv(join(DATA_DIR, "toots_moderated_instances.csv"))
moderated_toots = moderated_toots[["id", "content", "url"]]
moderated_toots["url_cleaned"] = moderated_toots.url.apply(clean_url)
moderated_toots["content_cleaned"] = moderated_toots["content"].map(clean_toots)
moderated_toots_sample = moderated_toots.sample(50000, random_state=27)
moderated_toots_sample.to_csv(join("openai_input", "sample_moderated_instances.csv"))

In [8]:
moderated_toots_sample.drop_duplicates(subset=["id"], inplace=True)

In [9]:
moderated_toots_sample.dropna(subset=["content_cleaned"], inplace=True)
moderated_toots_sample = moderated_toots_sample[moderated_toots_sample["content_cleaned"].astype(str).str.strip() != ""]

In [10]:
len(moderated_toots_sample)

47246

In [ ]:
# Serialize into jsonl
prepare_batch_input(moderated_toots, batch_no="moderated_servers_sample", text_col="cleaned_content", id_col="instance_id")

## Batches for mass-aggregated hate estimation

In [26]:
df.columns

Index(['id', 'instance', 'cleaned_content', 'instance_id'], dtype='object')

In [27]:
# Exclude toots that are already in the random sample to avoid running inference twice
df_rest = df[~df["instance_id"].isin(df_sampled["instance_id"])]

In [28]:
len(df_rest) + len(df_sampled) == len(df)

True

In [29]:
len(df_rest)

10818117

In [31]:
del df

In [35]:
df_rest.sort_values(by="instance", inplace=True)

In [36]:
batch_size = 49950
counter = 0
for start in range(0, len(df_rest), batch_size):
    df_batch = df_rest.iloc[start : start + batch_size]
    if df_batch.instance_id.nunique() != len(df_batch):
        raise ValueError("Duplicate instance_id in batch!")
    prepare_batch_input(df_batch, batch_no=counter, text_col="cleaned_content", id_col="instance_id", prefix="all_posts")
    print(f"Prepared batch {counter} with {len(df_batch)} toots.")
    counter += 1

Prepared batch 0 with 49950 toots.
Prepared batch 1 with 49950 toots.
Prepared batch 2 with 49950 toots.
Prepared batch 3 with 49950 toots.
Prepared batch 4 with 49950 toots.
Prepared batch 5 with 49950 toots.
Prepared batch 6 with 49950 toots.
Prepared batch 7 with 49950 toots.
Prepared batch 8 with 49950 toots.
Prepared batch 9 with 49950 toots.
Prepared batch 10 with 49950 toots.
Prepared batch 11 with 49950 toots.
Prepared batch 12 with 49950 toots.
Prepared batch 13 with 49950 toots.
Prepared batch 14 with 49950 toots.
Prepared batch 15 with 49950 toots.
Prepared batch 16 with 49950 toots.
Prepared batch 17 with 49950 toots.
Prepared batch 18 with 49950 toots.
Prepared batch 19 with 49950 toots.
Prepared batch 20 with 49950 toots.
Prepared batch 21 with 49950 toots.
Prepared batch 22 with 49950 toots.
Prepared batch 23 with 49950 toots.
Prepared batch 24 with 49950 toots.
Prepared batch 25 with 49950 toots.
Prepared batch 26 with 49950 toots.
Prepared batch 27 with 49950 toots.
Pr